In [1]:
import re
import json
import joblib
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore') # Ẩn các cảnh báo lặt vặt

In [2]:
ROOT_DIR = Path().resolve().parent

RAW_DATA_PATH = ROOT_DIR / "/kaggle/input/datasets/rmisra/news-category-dataset/News_Category_Dataset_v3.json"
PROCESSED_OUTPUT = ROOT_DIR / "/kaggle/working/news_processed.csv"
SPLITS_DIR = ROOT_DIR / "/kaggle/working/data/splits"

MODEL_OUTPUT = ROOT_DIR / "/kaggle/working/artifacts/models/candidates/baseline_tfidf_logreg.joblib"
METRICS_OUTPUT = ROOT_DIR / "/kaggle/working/artifacts/models/candidates/baseline_metrics.json"

# Tạo sẵn các thư mục nếu chưa tồn tại
PROCESSED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("Đã thiết lập xong đường dẫn!")

Đã thiết lập xong đường dẫn!


In [3]:
def clean_text(text: str) -> str:
    """Chuẩn hóa text: viết thường và xóa khoảng trắng thừa."""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def preprocess_dataset(input_path: Path) -> pd.DataFrame:
    """Load JSONL, gộp cột và làm sạch."""
    df = pd.read_json(input_path, lines=True)
    required_cols = ["headline", "short_description", "category"]
    
    df = df[required_cols].copy().dropna(subset=required_cols)
    df["text"] = (df["headline"].astype(str) + " " + df["short_description"].astype(str)).apply(clean_text)
    
    # Giữ lại các dòng hợp lệ
    df = df[(df["text"].str.len() > 0) & (df["category"].astype(str).str.len() > 0)]
    return df[["text", "category"]].reset_index(drop=True)

def make_splits(df: pd.DataFrame, test_size: float, val_size: float, random_state: int):
    """Chia tập train/val/test."""
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df["category"]
    )
    # Tính lại tỉ lệ val dựa trên phần còn lại
    val_ratio_from_train = val_size / (1.0 - test_size)
    train_df, val_df = train_test_split(
        train_df, test_size=val_ratio_from_train, random_state=random_state, stratify=train_df["category"]
    )
    return train_df, val_df, test_df

In [4]:
print(f"Đang đọc và tiền xử lý dữ liệu từ: {RAW_DATA_PATH}")
df = preprocess_dataset(RAW_DATA_PATH)

print("Đang chia tập train/val/test...")
train_df, val_df, test_df = make_splits(df, test_size=0.1, val_size=0.1, random_state=42)

# Lưu file
df.to_csv(PROCESSED_OUTPUT, index=False)
train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)

print(f"Tổng số dòng: {len(df):,}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
display(train_df.head(3))

Đang đọc và tiền xử lý dữ liệu từ: /kaggle/input/datasets/rmisra/news-category-dataset/News_Category_Dataset_v3.json
Đang chia tập train/val/test...
Tổng số dòng: 209,522
Train: 167,616 | Val: 20,953 | Test: 20,953


,text,category
186412,11 ways to exercise with your pet written by t...,WELLNESS
88629,these sisters are using their dance moves to e...,BLACK VOICES
87568,these 5 companies are trying to spark a parent...,BUSINESS


In [5]:
def load_split(path: Path) -> tuple[pd.Series, pd.Series]:
    """Load file CSV đã split."""
    df = pd.read_csv(path)
    return df["text"].astype(str), df["category"].astype(str)

def build_pipeline(max_features: int, min_df: int, max_df: float) -> Pipeline:
    """Xây dựng Pipeline TF-IDF + Logistic Regression."""
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), max_features=max_features,
            min_df=min_df, max_df=max_df, sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs", multi_class="auto")),
    ])

def evaluate(model: Pipeline, x: pd.Series, y: pd.Series) -> dict:
    """Tính toán metrics."""
    preds = model.predict(x)
    return {
        "accuracy": float(accuracy_score(y, preds)),
        "f1_macro": float(f1_score(y, preds, average="macro")),
        "report": classification_report(y, preds, output_dict=True, zero_division=0),
    }

In [6]:
print("Đang load dữ liệu từ splits...")
x_train, y_train = load_split(SPLITS_DIR / "train.csv")
x_val, y_val = load_split(SPLITS_DIR / "val.csv")
x_test, y_test = load_split(SPLITS_DIR / "test.csv")

print("Đang build và train pipeline (Có thể mất 1-2 phút)...")
model = build_pipeline(max_features=50000, min_df=3, max_df=0.95)
model.fit(x_train, y_train)

print("Đang đánh giá mô hình...")
val_metrics = evaluate(model, x_val, y_val)
test_metrics = evaluate(model, x_test, y_test)

# Lưu Model
joblib.dump(model, MODEL_OUTPUT)

# Lưu Metrics
metrics = {
    "model": "tfidf_logistic_regression_baseline",
    "validation": {"accuracy": val_metrics["accuracy"], "f1_macro": val_metrics["f1_macro"]},
    "test": {"accuracy": test_metrics["accuracy"], "f1_macro": test_metrics["f1_macro"]}
}
with METRICS_OUTPUT.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("✅ HOÀN TẤT!")
print(f"Mô hình lưu tại: {MODEL_OUTPUT}")
print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")

Đang load dữ liệu từ splits...
Đang build và train pipeline (Có thể mất 1-2 phút)...
Đang đánh giá mô hình...
✅ HOÀN TẤT!
Mô hình lưu tại: /kaggle/working/artifacts/models/candidates/baseline_tfidf_logreg.joblib
Validation Accuracy: 0.6033
Test Accuracy: 0.6043
